In [9]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from torchvision.models import VGG16_Weights

# ---------------------------------
# 1. Data Loading and Preprocessing
# ---------------------------------
def get_cifar10_dataloader():
    """
    Loads CIFAR-10 dataset with normalization and batching.

    Returns:
        DataLoader: PyTorch DataLoader for CIFAR-10.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4)
    return loader

# ------------------------
# Denormalization Function
# ------------------------
def denormalize(img_tensor):
    """
    Denormalizes CIFAR-10 images from [-1, 1] back to [0, 1].

    Args:
        img_tensor (Tensor): Normalized image tensor (B, 3, H, W)

    Returns:
        Tensor: Denormalized image tensor in [0, 1] range
    """
    mean = torch.tensor([0.4914, 0.4822, 0.4465], device=img_tensor.device).view(1,3,1,1)
    std = torch.tensor([0.247, 0.243, 0.261], device=img_tensor.device).view(1,3,1,1)
    img_tensor = img_tensor * std + mean
    img_tensor = torch.clamp(img_tensor, 0, 1)
    return img_tensor

# ----------------------------
# Create Masked Image Function
# ----------------------------
def create_masked_image(img_tensor, mask, patch_size):
    """
    Overlays gray patches on regions selected by mask.

    Args:
        img_tensor (Tensor): Original input image (B, 3, H, W)
        mask (Tensor): Binary mask of shape (B, N)
        patch_size (int): Patch size used for MAE

    Returns:
        Tensor: Masked image with gray patches
    """
    B, C, H, W = img_tensor.shape
    grid_size = H // patch_size  
    mask_2d = mask.reshape(B, grid_size, grid_size).unsqueeze(1).float()
    mask_full = F.interpolate(mask_2d, size=(H, W), mode='nearest')
    masked_img = img_tensor * (1 - mask_full) + mask_full * 0.5
    return masked_img

# ----------------------
# Patch Embedding Module
# ----------------------
class PatchEmbed(nn.Module):
    """
    Converts an image into a sequence of flattened patch embeddings.
    """
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)  # Shape: (B, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2).transpose(1, 2)  # Shape: (B, N, embed_dim)
        return x

# ----------------------------
# Transformer Block Definition
# ----------------------------
class TransformerBlock(nn.Module):
    """
    Basic transformer block with self-attention and MLP.
    """
    def __init__(self, embed_dim, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        # Apply LayerNorm and self-attention
        residual = x
        x = self.norm1(x)
        x_attn, _ = self.attn(x, x, x)
        x = residual + x_attn
        # Apply LayerNorm and MLP
        residual = x
        x = self.norm2(x)
        x = residual + self.mlp(x)
        return x

# --------------------
# MAE Model Definition
# --------------------
class MAE(nn.Module):
    """
    Masked Autoencoder with ViT encoder-decoder architecture.
    """
    def __init__(self, 
                 patch_size,
                 masking_ratio,
                 embed_dim=128,
                 decoder_embed_dim=128):
        super().__init__()
        self.masking_ratio = masking_ratio
        self.num_patches = (32 // patch_size) ** 2 
        self.patch_size = patch_size

        self.patch_embed = PatchEmbed( patch_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        
        self.encoder_blocks = nn.ModuleList([
            TransformerBlock(embed_dim) for _ in range(6)
        ])
        self.encoder_norm = nn.LayerNorm(embed_dim)
        
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.enc_to_dec = nn.Linear(embed_dim, decoder_embed_dim)
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, decoder_embed_dim))
        self.decoder_blocks = nn.ModuleList([
            TransformerBlock(decoder_embed_dim) for _ in range(4)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        
        patch_area = (patch_size ** 2) * 3
        self.recon_head = nn.Linear(decoder_embed_dim, patch_area)
        
        vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:16]
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg_feat = vgg.eval()
        
        self.initialize_weights()

    def initialize_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
    
    def random_masking(self, x):
        """
        Masks a random subset of patches for each sample in the batch.
        Returns the visible patches, binary mask, and restoration indices.
        """
        B, N, D = x.shape
        len_keep = int(N * (1 - self.masking_ratio))
        noise = torch.rand(B, N, device=x.device)

        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]

        x_visible = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        mask = torch.ones(B, N, device=x.device)
        mask.scatter_(1, ids_keep, 0)
        return x_visible, mask, ids_restore

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x) + self.pos_embed
        x_visible, mask, ids_restore = self.random_masking(x)

        for block in self.encoder_blocks:
            x_visible = block(x_visible)
        encoded = self.encoder_norm(x_visible)
        encoded = self.enc_to_dec(encoded)
        decoder_tokens = torch.zeros(B, self.num_patches, encoded.size(-1), device=x.device)
        
        # Insert visible tokens and mask tokens into decoder
        len_keep = encoded.size(1)
        decoder_tokens.scatter_(1, ids_restore[:, :len_keep].unsqueeze(-1).repeat(1, 1, encoded.size(-1)), encoded)
        mask_tokens = self.mask_token.expand(B, self.num_patches - len_keep, -1)
        decoder_tokens.scatter_(1, ids_restore[:, len_keep:].unsqueeze(-1).repeat(1, 1, encoded.size(-1)), mask_tokens)
        decoder_tokens += self.decoder_pos_embed

        for block in self.decoder_blocks:
            decoder_tokens = block(decoder_tokens)

        decoded = self.decoder_norm(decoder_tokens)
        x_recon_patches = self.recon_head(decoded)
        return x_recon_patches, mask

# ------------------------
# Perceptual Loss (Masked)
# ------------------------
def compute_masked_perceptual_loss(x_recon_patches, target_imgs, mask, patch_size, feature_extractor):
    """
    Computes perceptual loss using masked regions and VGG features.

    Returns:
        float: Normalized masked perceptual loss value
    """
    B = x_recon_patches.size(0)
    N = x_recon_patches.size(1)
    grid_size = int(math.sqrt(N))
    
    x_recon = x_recon_patches.reshape(B, grid_size, grid_size, -1).permute(0, 3, 1, 2)
    x_recon = x_recon.reshape(B, 3, grid_size * patch_size, grid_size * patch_size)
    
    mask_2d = mask.reshape(B, grid_size, grid_size).unsqueeze(1).float()
    mask_full = F.interpolate(mask_2d, size=(grid_size * patch_size, grid_size * patch_size), mode='nearest')
    
    features_recon = feature_extractor(x_recon)
    features_target = feature_extractor(target_imgs)
    mask_feat = F.interpolate(mask_full, size=features_recon.shape[-2:], mode='nearest')
    
    loss = F.mse_loss(features_recon * mask_feat, features_target * mask_feat, reduction='sum')
    norm_factor = mask_feat.sum() + 1e-6
    loss = loss / norm_factor
    return loss

# ----------------------
# Visualization Function
# ----------------------
def visualize_images(original, x_recon_patches, mask, patch_size, num_patches, epoch_size):
    """
    Plots original, masked, and reconstructed images.

    Args:
        original (Tensor): Original images
        x_recon_patches (Tensor): Predicted patch reconstructions
        mask (Tensor): Binary mask
    """
    grid_size = int(math.sqrt(num_patches))
    B = x_recon_patches.size(0)
    x_recon = x_recon_patches.reshape(B, grid_size, grid_size, -1).permute(0, 3, 1, 2)
    x_recon = x_recon.reshape(B, 3, grid_size * patch_size, grid_size * patch_size)
    masked_img = create_masked_image(original, mask, patch_size)
    
    original_np = denormalize(original).cpu().detach().numpy()
    masked_np = denormalize(masked_img).cpu().detach().numpy()
    recon_np = denormalize(x_recon).cpu().detach().numpy()
    
    fig, axes = plt.subplots(4, 3, figsize=(6, 12))
    for i in range(4):
        ax = axes[i, 0]
        ax.imshow(np.transpose(original_np[i], (1,2,0)))
        ax.set_title("Original")
        ax.axis("off")
        
        ax = axes[i, 1]
        ax.imshow(np.transpose(masked_np[i], (1,2,0)))
        ax.set_title("Masked")
        ax.axis("off")
        
        ax = axes[i, 2]
        ax.imshow(np.transpose(recon_np[i], (1,2,0)))
        ax.set_title("Reconstructed")
        ax.axis("off")

    plt.suptitle(f"Epoch {epoch_size} Results", fontsize=12)
    plt.tight_layout()
    plt.show()

# -------------
# Training Loop
# -------------
def train_mae(model, dataloader, optimizer, device, epoch_size):
    """
    Trains the MAE model and visualizes outputs periodically.
    """
    model.train()
    feature_extractor = model.vgg_feat.to(device)
    patch_size = model.patch_size
    total_training_time = 0.0

    for epoch in range(epoch_size):
        epoch_start = time.time()
        total_loss = 0.0
        for imgs, _ in dataloader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            x_recon_patches, mask = model(imgs)
            loss = compute_masked_perceptual_loss(x_recon_patches, imgs, mask, patch_size, feature_extractor)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        epoch_time = time.time() - epoch_start
        total_training_time += epoch_time
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{epoch_size}], Time: {epoch_time:.4f}s, Loss: {avg_loss:.4f}")
        
        if (epoch+1) % 10 == 0:
            visualize_images(imgs, x_recon_patches, mask, patch_size, model.num_patches, epoch+1)
    print(f"Total Training Time: {total_training_time:.4f}s")

In [10]:
def runMAE(patch_size, epoch_size, masking_ratios):
    """
    Runs the MAE training pipeline for multiple masking ratios using CIFAR-10.

    Args:
        patch_size (int): Patch size to divide the image into (e.g., 4, 8, 16).
        epoch_size (int): Number of epochs to train for each run.
        masking_ratios (list[int] or list[float]): List of masking ratios (as percentages) to evaluate.

    Example:
        runMAE(patch_size=4, epoch_size=20, masking_ratios=[25, 50, 75])
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader = get_cifar10_dataloader()
    start_time = time.time()
    for masking_ratio in masking_ratios:
        print(f'----------------------------------------- Masking Ratio: {masking_ratio}% --------------------------------------------------')
        model = MAE(patch_size, masking_ratio/100).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
        train_mae(model, train_loader, optimizer, device, epoch_size)
        print('============================================================================================================================')

    end_time = time.time() - start_time
    print('Total time', end_time)

In [ ]:
runMAE(16, 300, [25, 50, 75])

In [ ]:
runMAE(8, 200, [37.5, 50, 62.5])

In [ ]:
runMAE(8, 200, [62.5, 75])

In [ ]:
runMAE(8, 200, [75])

In [ ]:
runMAE(4, 100, [37.5, 50, 75])

In [ ]:
runMAE(4, 200, [75])

In [ ]:
runMAE(2, 10, [37.5, 50, 75])

In [ ]:
runMAE(2, 20, [50, 75])

In [ ]:
runMAE(2, 30, [75, 80])

In [ ]:
runMAE(2, 30, [85])